# Batch Gradient Descent

**Date:** 2026-04-26

---

## What is Batch Gradient Descent?

In **Batch Gradient Descent (BGD)**, we compute the gradient of the loss function using the **entire training dataset** before making a single parameter update.

### Update Rule

$$\theta = \theta - \alpha \cdot \frac{1}{m} \sum_{i=1}^{m} \nabla_{\theta} L(x_i, y_i)$$

- $m$ — total number of training samples
- $\alpha$ — learning rate
- One update happens **after** seeing all $m$ samples (= one epoch)

### Pros
- Stable, smooth loss curve — low variance gradient
- Guaranteed to converge to global minimum (convex problems)
- Easy to debug — loss decreases monotonically

### Cons
- Very slow for large datasets — only one update per full pass
- Needs entire dataset in memory
- Cannot be used for online learning

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 4)

## 1. Generate Data

We create data from `y = 3x + 2 + noise`. Goal: recover W=3, b=2 using Batch GD.

In [ ]:
m = 1000
X = np.random.randn(m, 1)
y = 3 * X + 2 + np.random.randn(m, 1) * 0.5

plt.scatter(X, y, alpha=0.3, s=10, color='steelblue')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Training Data  (y = 3x + 2 + noise)')
plt.show()

## 2. Helper Functions

**Loss (MSE):**
$$L = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)^2$$

**Gradients:**
$$\frac{\partial L}{\partial W} = \frac{2}{m} X^T (\hat{y} - y), \quad \frac{\partial L}{\partial b} = \frac{2}{m} \sum (\hat{y} - y)$$

In [ ]:
def predict(X, W, b):
    return X @ W + b

def mse_loss(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)

def compute_gradients(X, y, W, b):
    n = len(y)
    error = predict(X, W, b) - y
    dW = (2 / n) * X.T @ error
    db = (2 / n) * np.sum(error)
    return dW, db

## 3. Batch Gradient Descent — Implementation

Each epoch:
1. Pass **all m samples** through the model
2. Compute gradient over the full dataset
3. Make **one** parameter update

In [ ]:
def batch_gradient_descent(X, y, lr=0.1, epochs=200):
    W = np.zeros((X.shape[1], 1))
    b = 0.0
    history = {'loss': [], 'W': [], 'b': []}

    for epoch in range(epochs):
        # Gradient computed over ALL m samples
        dW, db = compute_gradients(X, y, W, b)

        # Single update per epoch
        W -= lr * dW
        b -= lr * db

        loss = mse_loss(predict(X, W, b), y)
        history['loss'].append(loss)
        history['W'].append(W[0, 0])
        history['b'].append(b)

    return W, b, history

W_final, b_final, history = batch_gradient_descent(X, y, lr=0.1, epochs=200)

print(f"Learned  ->  W = {W_final[0,0]:.4f},  b = {b_final:.4f}")
print(f"True     ->  W = 3.0000,  b = 2.0000")
print(f"Final MSE Loss: {history['loss'][-1]:.4f}")

## 4. Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curve
axes[0].plot(history['loss'], color='steelblue', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Loss Curve — Batch GD')
axes[0].grid(True, alpha=0.3)

# W convergence
axes[1].plot(history['W'], color='orange', linewidth=2)
axes[1].axhline(y=3, color='red', linestyle='--', label='True W=3')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('W')
axes[1].set_title('Weight W Convergence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# b convergence
axes[2].plot(history['b'], color='green', linewidth=2)
axes[2].axhline(y=2, color='red', linestyle='--', label='True b=2')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('b')
axes[2].set_title('Bias b Convergence')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Batch Gradient Descent', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Effect of Learning Rate on Batch GD

In [ ]:
learning_rates = [0.001, 0.01, 0.1, 0.5, 1.0]
colors = ['purple', 'blue', 'green', 'orange', 'red']

plt.figure(figsize=(12, 4))
for lr, color in zip(learning_rates, colors):
    _, _, hist = batch_gradient_descent(X, y, lr=lr, epochs=100)
    losses = np.clip(hist['loss'], 0, 30)
    plt.plot(losses, label=f'lr = {lr}', color=color, linewidth=2)

plt.xlabel('Epoch')
plt.ylabel('MSE Loss (clipped at 30)')
plt.title('Batch GD — Effect of Learning Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Observation:")
print("  lr=0.001 -> converges very slowly")
print("  lr=0.1   -> converges well (sweet spot)")
print("  lr=1.0   -> likely diverges (loss explodes)")

## 6. Key Observations

- **Smooth loss curve** — because we average gradients over all m=1000 samples, the update direction is very accurate
- **Monotonically decreasing** loss — no noise, no oscillations
- **One update per epoch** — with large datasets this is expensive; for m=1,000,000 samples you wait for 1M forward passes before a single weight update
- **Sensitive to learning rate** — too high diverges, too low is painfully slow

---

**Next:** `02_stochastic_gradient_descent.ipynb`